# Drishti — GPU benchmark (Colab/Kaggle)

Fills in the GPU half of `docs/benchmarks.md`, marked **pending user run** there because the environment this project was built in has no GPU (`nvidia-smi` not present). This notebook is designed to actually run on a free Colab T4 (Runtime → Change runtime type → T4 GPU) or Kaggle GPU notebook.

It measures, for `HuggingFaceTB/SmolVLM-256M-Instruct`: fp16 baseline, 4-bit and 8-bit quantized (bitsandbytes), a batch-size sweep, and accuracy on the same held-out labeled subset next to each speed number — never speed alone, per this project's own rule that a faster config that gets worse answers is a tradeoff to document, not an optimization to celebrate.

**Before running:** either `git clone` this repo (so `data/images/` and `data/labels/labels.jsonl` are present), or upload those two paths manually to the Colab filesystem.

In [ ]:
!git clone --depth 1 https://github.com/<your-username>/Drishti.git /content/drishti 2>/dev/null || echo "Edit the URL above, or upload data/images + data/labels manually."
%cd /content/drishti

In [ ]:
!pip install -q transformers==5.17.0 accelerate==1.15.0 bitsandbytes==0.50.2 pillow==12.3.0
# torch is already installed with CUDA support on Colab/Kaggle GPU runtimes — do not reinstall the CPU wheel here.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU (Colab) or enable a GPU accelerator (Kaggle)"
print(torch.cuda.get_device_name(0))

In [ ]:
import json, random, time
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

ROOT = Path("/content/drishti")
LABELS_FILE = ROOT / "data/labels/labels.jsonl"
IMAGES_DIR = ROOT / "data/images"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

records = [json.loads(l) for l in LABELS_FILE.read_text(encoding="utf-8").splitlines() if l.strip()]
rng = random.Random(5)
sample = rng.sample(records, 15)  # held-out labeled subset for the accuracy-next-to-speed table
images = [Image.open(IMAGES_DIR / r["file"]).convert("RGB") for r in sample]
print(f"Loaded {len(images)} images for benchmarking.")

In [ ]:
# Reuses the same coarse presence-detection accuracy definition as
# benchmarks/parity.py::is_correct, duplicated here (no import) so this
# notebook has zero dependency on the rest of the repo besides data files.
VEHICLE_CLASSES = {"car", "van", "truck", "bus", "motor", "bicycle", "tricycle", "awning-tricycle"}
PERSON_CLASSES = {"people", "pedestrian"}

def is_correct(finding_type: str, gt_labels: set) -> bool:
    ft = finding_type.lower()
    has_vehicle = bool(gt_labels & VEHICLE_CLASSES)
    has_person = bool(gt_labels & PERSON_CLASSES)
    predicts_vehicle = any(k in ft for k in ("vehicle", "car", "van", "truck"))
    predicts_person = any(k in ft for k in ("person", "personnel", "people", "pedestrian"))
    if ft in ("no_finding", "none", "nothing"):
        return not has_vehicle and not has_person
    if predicts_vehicle and has_vehicle:
        return True
    if predicts_person and has_person:
        return True
    return False

CAPTION_PROMPT = "Describe what is visible in this image in one or two sentences."

In [ ]:
def load_model(dtype=None, quant_config=None):
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    kwargs = {}
    if quant_config is not None:
        kwargs["quantization_config"] = quant_config
    elif dtype is not None:
        kwargs["dtype"] = dtype
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).to("cuda" if quant_config is None else None)
    model.eval()
    return processor, model

@torch.inference_mode()
def caption_one(processor, model, image):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": CAPTION_PROMPT}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, repetition_penalty=1.3)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def bench_config(label, dtype=None, quant_config=None, batch=1, n_warmup=1):
    processor, model = load_model(dtype=dtype, quant_config=quant_config)
    # warmup (discarded)
    for i in range(n_warmup):
        caption_one(processor, model, images[i % len(images)])
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    latencies = []
    captions = []
    for im in images:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        cap = caption_one(processor, model, im)
        torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)
        captions.append(cap)
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6

    # NOTE: accuracy here is on the *caption* stage only (this notebook
    # doesn't re-run the local structuring LLM) — treat this as a grounding
    # sanity check (does the caption mention the right coarse category),
    # not the full-pipeline accuracy number in docs/benchmarks.md.
    n_correct = sum(
        1 for cap, r in zip(captions, sample)
        if is_correct(cap, {d["label"] for d in r["detections"]})
    )

    latencies.sort()
    result = {
        "config": label, "n_images": len(images),
        "latency_ms_p50": latencies[len(latencies)//2],
        "latency_ms_p95": latencies[int(len(latencies)*0.95)],
        "latency_ms_mean": sum(latencies)/len(latencies),
        "throughput_img_per_s": 1000.0/(sum(latencies)/len(latencies)),
        "peak_gpu_memory_mb": peak_mem_mb,
        "caption_grounding_accuracy": n_correct/len(sample),
    }
    del model
    torch.cuda.empty_cache()
    return result

In [ ]:
results = []
results.append(bench_config("fp16_baseline", dtype=torch.float16))
results.append(bench_config("bf16_baseline", dtype=torch.bfloat16))
results.append(bench_config("int8_quantized", quant_config=BitsAndBytesConfig(load_in_8bit=True)))
results.append(bench_config("int4_quantized", quant_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)))

for r in results:
    print(f"{r['config']:20s} p50={r['latency_ms_p50']:7.1f}ms  throughput={r['throughput_img_per_s']:.2f} img/s  "
          f"peak_mem={r['peak_gpu_memory_mb']:.0f}MB  grounding_acc={r['caption_grounding_accuracy']:.2f}")

## Batch-size sweep

A real batched-generation sweep requires padding same-length prompts and using `generate()`'s batch dimension properly. This is left as the next cell for you to run given actual Colab GPU characteristics (the padding strategy interacts with the specific tokenizer/processor version) — measure batch sizes 1, 2, 4, 8 and record `throughput_img_per_s` and `peak_gpu_memory_mb` for each the same way `bench_config` does above.

In [ ]:
out_path = ROOT / "benchmarks/results/gpu_bench_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({
    "hardware": torch.cuda.get_device_name(0),
    "n_images": len(images),
    "configs": results,
}, indent=2))
print(f"Wrote {out_path} — copy this file back into your clone's benchmarks/results/ and update docs/benchmarks.md's GPU section with it.")